# Red RVC Voice Conversion — Simple Colab
Use a GPU runtime (T4 if available). Run the single cell below. It installs CoverGen-RVC, downloads its support models, downloads your Red model, and launches the interface.

In [ ]:
import os, subprocess, sys, shutil

REPO="/content/CoverGen"
MODEL_DIR=f"{REPO}/rvc_models/Red"
MODEL=f"{MODEL_DIR}/RedsVoiceSwap_53e_424s.pth"
DRIVE_ID="19yLeLybGU8csalpLFuK6ORSS3aqaDkrW"

print("1/6 Cleaning old runtime...")
shutil.rmtree(REPO, ignore_errors=True)
print("2/6 Getting CoverGen-RVC...")
subprocess.run(["git","lfs","install"], check=True)
subprocess.run(["git","clone","https://huggingface.co/spaces/justyoung/CoverGen-RVC",REPO], check=True)
os.chdir(REPO)
print("3/6 Installing dependencies...")
subprocess.run([sys.executable,"-m","pip","install","pip==23.1"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","gdown"], check=True)
print("4/6 Downloading RVC support models...")
subprocess.run([sys.executable,"src/download_models.py"], check=True)
print("5/6 Downloading your Red voice model...")
os.makedirs(MODEL_DIR, exist_ok=True)
subprocess.run([sys.executable,"-m","gdown",f"https://drive.google.com/uc?id={DRIVE_ID}","-O",MODEL], check=True)
if not os.path.isfile(MODEL) or os.path.getsize(MODEL) < 50_000_000:
    raise RuntimeError("Red voice model download failed or is incomplete.")
print(f"Red model ready: {os.path.getsize(MODEL)/1024/1024:.1f} MB")
print("6/6 Checking GPU...")
subprocess.run([sys.executable,"-c","import torch; print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"],check=True)
print("Launching CoverGen-RVC. Wait for the public Gradio URL below.")
subprocess.run([sys.executable,"src/covergen.py","False"],check=True)
